In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix

VOCAB_SIZE = 10000
MAX_LEN = 200

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Test samples: 25000
X_train shape: (25000, 200)
X_test shape: (25000, 200)


In [5]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=128),
    layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,331,521 (5.08 MB)

 Trainable params: 1,331,521 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 240s 727ms/step - accuracy: 0.5331 - loss: 0.6852 - val_accuracy: 0.6090 - val_loss: 0.6453
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 221s 706ms/step - accuracy: 0.5634 - loss: 0.6699 - val_accuracy: 0.6102 - val_loss: 0.6273
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 220s 703ms/step - accuracy: 0.6381 - loss: 0.6042 - val_accuracy: 0.7710 - val_loss: 0.5953
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 222s 710ms/step - accuracy: 0.8169 - loss: 0.4426 - val_accuracy: 0.8136 - val_loss: 0.4421
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 223s 714ms/step - accuracy: 0.8496 - loss: 0.3883 - val_accuracy: 0.8518 - val_loss: 0.3787


In [7]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=['Negative', 'Positive']
    )
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

model.save("imdb_lstm.h5")
print("\nModel saved as imdb_lstm.h5")

782/782 ━━━━━━━━━━━━━━━━━━━━ 104s 132ms/step

Confusion Matrix:
[[10244  2256]
 [ 1876 10624]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.85      0.82      0.83     12500
    Positive       0.82      0.85      0.84     12500

    accuracy                           0.83     25000
   macro avg       0.84      0.83      0.83     25000
weighted avg       0.84      0.83      0.83     25000




Test Accuracy: 83.47%

Model saved as imdb_lstm.h5


In [9]:
model.save("imdb_lstm.h5")

In [10]:
import os

print(os.path.abspath("imdb_lstm.h5"))

/content/imdb_lstm.h5
